# PitchsideIQ: Exploratory Data Analysis (EDA)
### Statistical Analysis of 20 Years of European Football Dynamics (2005–2025)

**Project Goal:** Uncover underlying behavioral patterns, test football hypotheses, quantify the legendary "Home Advantage," analyze scoring distributions, and evaluate betting market efficiency.

#### Key Hypotheses & Exploration Angles:
1. **The Home Advantage Phenomenon:** Does the home team really enjoy a significant statistical edge, and has it changed over 20 seasons?
2. **Poisson Scoring Dynamics:** Are match scorelines Poisson-distributed? What are the goal frequencies across Europe's top 5 leagues?
3. **The 2020 COVID Empty Stadiums Natural Experiment:** Did playing behind closed doors without home fans eliminate the home advantage?
4. **Shot Conversion & Attacking Dominance:** How strongly do Total Shots and Shots on Target correlate with goals scored?
5. **Dynamic Elo Separation:** Does a higher Elo rating reliably predict victory across diverse competitions?
6. **Market Efficiency & Odds Calibration:** How well do bookmaker odds reflect real-world probability outcomes?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("EDA environment ready!")

EDA environment ready!


## 1. Load Processed Match Dataset
We load `processed_data.csv` which contains 20 seasons of standardized match records and pre-match analytical features.

In [3]:
data_path = Path("processed_data.csv")
if not data_path.exists():
    data_path = Path("data/processed_data.csv")

df = pd.read_csv(data_path, parse_dates=["MatchDate"])
print(f"Loaded {len(df):,} matches across {df['LeagueCode'].nunique()} leagues.")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data\\processed_data.csv'

## 2. Match Outcome Distribution & Home Advantage Quantification
In football theory, the host club benefits from familiar pitch dimensions, local crowd noise influencing referee decisions, and zero travel fatigue.
Let's measure the historical outcome distribution.

In [4]:
outcome_counts = df['FTResult'].value_counts(normalize=True) * 100
labels = ['Home Win (H)', 'Away Win (A)', 'Draw (D)']
colors = ['#10b981', '#ef4444', '#f59e0b']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
sns.barplot(x=outcome_counts.index, y=outcome_counts.values, palette=colors, ax=ax1)
ax1.set_title("Overall Match Outcome Distribution (%)", fontsize=13, fontweight='bold')
ax1.set_ylabel("Percentage of Matches (%)")
for p in ax1.patches:
    ax1.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() - 4),
                 ha='center', va='center', color='white', fontweight='bold', fontsize=12)

# Pie chart
ax2.pie(outcome_counts, labels=labels, autopct='%1.1f%%', colors=colors, startangle=140,
        textprops={'fontsize': 11, 'fontweight': 'bold'})
ax2.set_title("Home Advantage Visual Share", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

NameError: name 'df' is not defined

## 3. Scoring Distributions & League-by-League Comparison
Are goals Poisson distributed? How do scoring rates vary between leagues (e.g. offensive Bundesliga vs tactical Serie A)?

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Home Goals vs Away Goals distribution
bins = np.arange(0, 10) - 0.5
ax1.hist(df['FTHome'], bins=bins, alpha=0.6, label='Home Goals', color='#2563eb', density=True)
ax1.hist(df['FTAway'], bins=bins, alpha=0.6, label='Away Goals', color='#dc2626', density=True)
ax1.set_title("Distribution of Goals per Match (Home vs Away)", fontsize=13, fontweight='bold')
ax1.set_xlabel("Goals Scored")
ax1.set_ylabel("Probability Density")
ax1.set_xticks(range(0, 9))
ax1.legend(fontsize=11)

# Average goals per match by League
df['TotalGoals'] = df['FTHome'] + df['FTAway']
league_goals = df.groupby('LeagueCode')['TotalGoals'].mean().sort_values(ascending=False)
sns.barplot(x=league_goals.index, y=league_goals.values, palette='viridis', ax=ax2)
ax2.set_title("Average Total Goals per Match by League", fontsize=13, fontweight='bold')
ax2.set_ylabel("Average Goals")
for p in ax2.patches:
    ax2.annotate(f"{p.get_height():.2f}", (p.get_x() + p.get_width() / 2., p.get_height() - 0.25),
                 ha='center', va='center', color='white', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

## 4. Evolution of Home Advantage Over Time: The 2020 COVID Natural Experiment
During the 2020–2021 pandemic season, matches were played in empty stadiums ("Ghost Games").
Did the absence of fans diminish the home advantage?

In [ ]:
yearly_outcomes = df.groupby('Year')['FTResult'].value_counts(normalize=True).unstack() * 100

plt.figure(figsize=(14, 5))
plt.plot(yearly_outcomes.index, yearly_outcomes['H'], marker='o', linewidth=2.5, color='#10b981', label='Home Win %')
plt.plot(yearly_outcomes.index, yearly_outcomes['A'], marker='s', linewidth=2.5, color='#ef4444', label='Away Win %')
plt.plot(yearly_outcomes.index, yearly_outcomes['D'], marker='^', linewidth=2.5, color='#f59e0b', label='Draw %')

plt.axvline(x=2020, color='gray', linestyle='--', linewidth=1.5, alpha=0.8, label='COVID-19 Empty Stadiums (2020)')
plt.title("Evolution of Home Advantage Across 20 Seasons (2005–2025)", fontsize=13, fontweight='bold')
plt.xlabel("Year")
plt.ylabel("Match Outcome Share (%)")
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 5. Attacking Efficiency: Shots vs Shots on Target vs Goals
Does generating more shots guarantee more goals, or is shot accuracy (Shots on Target) the deciding factor?

In [ ]:
if 'HomeShotsTarget' in df.columns and 'HomeShots' in df.columns:
    sample_df = df.dropna(subset=['HomeShots', 'HomeShotsTarget', 'FTHome']).sample(min(3000, len(df)), random_state=42)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    sns.regplot(data=sample_df, x='HomeShots', y='FTHome', scatter_kws={'alpha':0.2, 'color':'#2563eb'}, line_kws={'color':'blue'}, ax=ax1)
    ax1.set_title("Total Home Shots vs Goals Scored", fontsize=12, fontweight='bold')
    ax1.set_ylabel("Home Goals")
    
    sns.regplot(data=sample_df, x='HomeShotsTarget', y='FTHome', scatter_kws={'alpha':0.2, 'color':'#10b981'}, line_kws={'color':'green'}, ax=ax2)
    ax2.set_title("Home Shots on Target vs Goals Scored", fontsize=12, fontweight='bold')
    ax2.set_ylabel("Home Goals")
    
    plt.tight_layout()
    plt.show()

## 6. Dynamic Elo Ratings & Outcome Separation
Elo differences quantify the structural skill gap between two clubs.
Let's see how `EloDiff` distributions separate by final match outcome.

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='FTResult', y='EloDiff', palette={'H': '#10b981', 'D': '#f59e0b', 'A': '#ef4444'},
            order=['H', 'D', 'A'])
plt.title("Elo Rating Difference (Home Elo - Away Elo) by Outcome", fontsize=13, fontweight='bold')
plt.xlabel("Full-Time Result (H=Home Win, D=Draw, A=Away Win)")
plt.ylabel("Pre-Match Elo Difference")
plt.axhline(0, color='black', linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

## 7. Market Calibration: Implied Odds Probability vs Observed Outcomes
In efficient betting markets, if a bookmaker gives an event an implied probability of 60%, it should occur approximately 60% of the time.

In [ ]:
valid_odds = df.dropna(subset=['ImpProbHome', 'FTResult']).copy()
valid_odds['ProbBin'] = pd.cut(valid_odds['ImpProbHome'], bins=np.linspace(0.1, 0.9, 9))
calibration = valid_odds.groupby('ProbBin', observed=True).agg(
    Avg_Predicted=('ImpProbHome', 'mean'),
    Actual_Win_Rate=('FTResult', lambda r: (r == 'H').mean())
).reset_index()

plt.figure(figsize=(7, 6))
plt.plot([0.1, 0.9], [0.1, 0.9], 'k--', label='Perfect Calibration (y = x)')
plt.plot(calibration['Avg_Predicted'], calibration['Actual_Win_Rate'], marker='o', color='#2563eb', linewidth=2.5, label='Observed Win Rate')
plt.title("Bookmaker Market Probability Calibration Curve", fontsize=13, fontweight='bold')
plt.xlabel("Bookmaker Implied Probability (1 / Odds)")
plt.ylabel("Observed Home Win Proportion")
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 8. Correlation Matrix of Analytical Features
Examining pairwise correlations between dynamic pre-match signals and outcomes.

In [ ]:
num_cols = ['EloDiff', 'AdjFormDiff', 'TSRDiff', 'RestDiff', 'ImpProbHome', 'ImpProbAway', 'FTHome', 'FTAway', 'outcome_1x2']
corr_matrix = df[num_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', cbar=True, square=True)
plt.title("Correlation Matrix of Pre-Match Features & Outcomes", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()